# 🐘 İleri Veri Mühendisliği — Bölüm 1: Apache Hadoop Uygulamalı Lab (Google Colab)

Bu not defteri Colab makinesinde **tek düğümlü (pseudo-distributed) gerçek bir Hadoop kümesi** kurar: NameNode, DataNode, ResourceManager, NodeManager ve JobHistory Server ayrı JVM süreçleri olarak çalışır. HTML rehberdeki kavramları komut ve kodla deneyeceğiz.

| # | Konu |
|---|------|
| 0 | Kurulum: Java 11, Hadoop indirme, yapılandırma, servisleri başlatma, Web arayüzleri |
| 1 | Lab verisi üretimi (metin korpusu, erişim logları, kullanıcı/sipariş tabloları) |
| 2 | HDFS temelleri: dosya komutları, bloklar, `fsck`, DataNode diskindeki fiziksel bloklar |
| 3 | HDFS ileri: replikasyon, çöp kutusu, snapshot, kota, küçük dosya problemi, HAR, WebHDFS |
| 4 | MapReduce: örnek işler, Java WordCount (combiner + 2 reducer) |
| 5 | Hadoop Streaming (Python): log analizi, combiner tuzağı, reduce-side join |
| 6 | YARN: düğümler, uygulamalar, loglar, kuyruklar, REST API |
| 7 | Dosya formatları: CSV vs Parquet vs ORC |
| 8 | Köprü: Spark ile HDFS'ten okuma/yazma |
| 9 | Kapatma ve alıştırmalar |

> ⏳ İlk kurulum (Hadoop indirmesi ~500 MB) Colab'da genellikle 2–5 dakika sürer. Sonraki hücreler hızlıdır. Colab oturumu kapanınca küme de silinir; yeniden bağlanırsanız baştan çalıştırın.

## 0. Kurulum
### 0.1 Ortam değişkenleri ve Java 11

In [ ]:
import os, sys, subprocess, glob, time, textwrap

IN_COLAB = "google.colab" in sys.modules
HADOOP_VERSION = "3.4.1"

os.environ.update({
    "HADOOP_VERSION": HADOOP_VERSION,
    "JAVA_HOME":      "/usr/lib/jvm/java-11-openjdk-amd64",
    "HADOOP_HOME":    "/usr/local/hadoop",
    "HADOOP_CONF_DIR": "/usr/local/hadoop/etc/hadoop",
    "HADOOP_DATA":    "/opt/hadoop_data",                 # NameNode/DataNode yerel dizinleri
    "LAB":            "/content/hadoop_lab" if IN_COLAB else os.path.expanduser("~/hadoop_lab"),
    # Colab'da root olarak çalışıyoruz; Hadoop 3 daemon'ları bunun açıkça belirtilmesini ister
    "HDFS_NAMENODE_USER": "root", "HDFS_DATANODE_USER": "root", "HDFS_SECONDARYNAMENODE_USER": "root",
    "YARN_RESOURCEMANAGER_USER": "root", "YARN_NODEMANAGER_USER": "root", "MAPRED_HISTORYSERVER_USER": "root",
})
os.environ["PATH"] = (f'{os.environ["HADOOP_HOME"]}/bin:{os.environ["HADOOP_HOME"]}/sbin:'
                      f'{os.environ["JAVA_HOME"]}/bin:' + os.environ["PATH"])
os.makedirs(os.environ["LAB"], exist_ok=True)

# Hadoop 3.4 Java 8 ve 11'i resmi olarak destekler. Spark köprüsü (Bölüm 8) için Java 17'yi de kuruyoruz.
subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-11-jdk-headless openjdk-17-jdk-headless > /dev/null",
               shell=True, check=True)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])

### 0.2 Hadoop'u indir

In [ ]:
%%bash
set -e
if [ -x "$HADOOP_HOME/bin/hadoop" ]; then echo "Hadoop zaten kurulu."; exit 0; fi
cd /tmp
# 'lean' paket AWS SDK'sını içermez (~yarı boyut). Ayna çalışmazsa arşive ve tam pakete düşer.
for URL in \
  "https://dlcdn.apache.org/hadoop/common/hadoop-${HADOOP_VERSION}/hadoop-${HADOOP_VERSION}-lean.tar.gz" \
  "https://archive.apache.org/dist/hadoop/common/hadoop-${HADOOP_VERSION}/hadoop-${HADOOP_VERSION}-lean.tar.gz" \
  "https://archive.apache.org/dist/hadoop/common/hadoop-${HADOOP_VERSION}/hadoop-${HADOOP_VERSION}.tar.gz"; do
  echo "İndiriliyor: $URL"
  if wget -q -O hadoop.tgz "$URL" && tar -tzf hadoop.tgz > /dev/null 2>&1; then OK=1; break; fi
done
[ "$OK" = "1" ] || { echo "İndirme başarısız!"; exit 1; }
DIR=$(tar -tzf hadoop.tgz | head -1 | cut -d/ -f1)
tar -xzf hadoop.tgz -C /usr/local && rm hadoop.tgz
mv "/usr/local/$DIR" "$HADOOP_HOME"
hadoop version | head -1

### 0.3 Yapılandırma (pseudo-distributed)
HTML rehberdeki dört temel dosyayı yazıyoruz. Tek DataNode olduğu için `dfs.replication=1`. Colab belleğine göre YARN'a 6 GB / 4 vCore veriyoruz.

In [ ]:
HH, CONF, DATA = os.environ["HADOOP_HOME"], os.environ["HADOOP_CONF_DIR"], os.environ["HADOOP_DATA"]

def write_xml(name, props):
    body = "\n".join(f"  <property>\n    <name>{k}</name>\n    <value>{v}</value>\n  </property>"
                     for k, v in props.items())
    with open(f"{CONF}/{name}", "w") as f:
        f.write(f'<?xml version="1.0"?>\n<configuration>\n{body}\n</configuration>\n')
    print(f"✔ {name}: {len(props)} özellik")

write_xml("core-site.xml", {
    "fs.defaultFS": "hdfs://localhost:9000",
    "hadoop.tmp.dir": f"{DATA}/tmp",
    "fs.trash.interval": "60",                       # silinen dosyalar 60 dk çöp kutusunda kalır
    "hadoop.http.staticuser.user": "root",           # Web UI / WebHDFS kullanıcısı
})
write_xml("hdfs-site.xml", {
    "dfs.replication": "1",
    "dfs.namenode.name.dir": f"file://{DATA}/name",
    "dfs.datanode.data.dir": f"file://{DATA}/data",
    "dfs.blocksize": "134217728",                    # 128 MB (varsayılan)
    "dfs.permissions.enabled": "false",
    "dfs.namenode.safemode.extension": "0",
})
write_xml("mapred-site.xml", {
    "mapreduce.framework.name": "yarn",
    "mapreduce.application.classpath":
        "$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/*:$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/lib/*",
    "yarn.app.mapreduce.am.env": f"HADOOP_MAPRED_HOME={HH}",
    "mapreduce.map.env": f"HADOOP_MAPRED_HOME={HH}",
    "mapreduce.reduce.env": f"HADOOP_MAPRED_HOME={HH}",
    "yarn.app.mapreduce.am.resource.mb": "1024",
    "mapreduce.map.memory.mb": "1024",
    "mapreduce.reduce.memory.mb": "1024",
    "mapreduce.map.java.opts": "-Xmx800m",
    "mapreduce.reduce.java.opts": "-Xmx800m",
    "mapreduce.map.output.compress": "true",          # ara çıktıyı sıkıştır (shuffle trafiği azalır)
})
write_xml("yarn-site.xml", {
    "yarn.resourcemanager.hostname": "localhost",
    "yarn.nodemanager.aux-services": "mapreduce_shuffle",
    "yarn.nodemanager.env-whitelist":
        "JAVA_HOME,HADOOP_COMMON_HOME,HADOOP_HDFS_HOME,HADOOP_CONF_DIR,CLASSPATH_PREPEND_DISTCACHE,"
        "HADOOP_YARN_HOME,HADOOP_HOME,PATH,LANG,TZ,HADOOP_MAPRED_HOME",
    "yarn.nodemanager.resource.memory-mb": "6144",
    "yarn.nodemanager.resource.cpu-vcores": "4",
    "yarn.scheduler.minimum-allocation-mb": "512",
    "yarn.scheduler.maximum-allocation-mb": "4096",
    "yarn.nodemanager.vmem-check-enabled": "false",   # Colab'da sanal bellek kontrolü container'ları öldürebilir
    "yarn.nodemanager.disk-health-checker.max-disk-utilization-per-disk-percentage": "99",
    "yarn.log-aggregation-enable": "true",            # 'yarn logs' için
})
# Capacity Scheduler: iki kuyruk (default %60, etl %40)
write_xml("capacity-scheduler.xml", {
    "yarn.scheduler.capacity.maximum-applications": "100",
    "yarn.scheduler.capacity.maximum-am-resource-percent": "0.5",   # küçük kümede AM'ler kuyrukta takılmasın
    "yarn.scheduler.capacity.resource-calculator":
        "org.apache.hadoop.yarn.util.resource.DefaultResourceCalculator",
    "yarn.scheduler.capacity.root.queues": "default,etl",
    "yarn.scheduler.capacity.root.default.capacity": "60",
    "yarn.scheduler.capacity.root.default.maximum-capacity": "100",
    "yarn.scheduler.capacity.root.etl.capacity": "40",
    "yarn.scheduler.capacity.root.etl.maximum-capacity": "80",
    "yarn.scheduler.capacity.root.default.state": "RUNNING",
    "yarn.scheduler.capacity.root.etl.state": "RUNNING",
})

with open(f"{CONF}/hadoop-env.sh", "a") as f:
    f.write(f'\nexport JAVA_HOME={os.environ["JAVA_HOME"]}\n')

### 0.4 NameNode'u biçimlendir ve servisleri başlat

In [ ]:
%%bash
if [ ! -d "$HADOOP_DATA/name/current" ]; then
  hdfs namenode -format -force -nonInteractive > /tmp/nn_format.log 2>&1 && echo "✔ NameNode biçimlendirildi"
fi
hdfs  --daemon start namenode
hdfs  --daemon start datanode
yarn  --daemon start resourcemanager
yarn  --daemon start nodemanager
mapred --daemon start historyserver
hdfs dfsadmin -safemode wait
sleep 8
echo "--- Çalışan Java süreçleri (jps) ---"
jps | grep -v Jps | sort -k2

### 0.5 Web arayüzleri
Aşağıdaki bağlantılar Colab'da yeni sekmede açılır:
- **9870** → NameNode UI (Datanodes, Browse the file system, Snapshot, Startup progress)
- **8088** → YARN ResourceManager UI (Applications, Scheduler/kuyruklar, Nodes)
- **19888** → MapReduce JobHistory (biten işlerin counter'ları, map/reduce süreleri)

> İçerideki bazı bağlantılar `localhost` adını kullandığından açılmayabilir; ana sayfalar çalışır.

In [ ]:
if IN_COLAB:
    from google.colab import output
    for port, path in [(9870, "/"), (8088, "/cluster"), (19888, "/jobhistory")]:
        output.serve_kernel_port_as_window(port, path=path)
else:
    print("http://localhost:9870  http://localhost:8088  http://localhost:19888")

## 1. Lab Verisi
- `corpus.txt` (~45 MB): Zipf dağılımlı Türkçe kelimelerden oluşan metin → WordCount
- `access.log` (~400K satır): web sunucusu erişim logları → Streaming log analizi
- `users.csv`, `orders.csv` → reduce-side join
- `small/` : 1.500 küçük dosya → küçük dosya problemi

In [ ]:
import random, datetime
random.seed(42)
LAB = os.environ["LAB"]

vocab = ("veri hadoop spark küme düğüm blok replika namenode datanode yarn kaynak kuyruk görev "
         "harita indirgeme anahtar değer bölüm karıştırma sıralama birleştirme dosya dizin disk ağ "
         "bellek işlemci ölçek hata tolerans analitik akış tablo sorgu şema format sıkıştırma "
         "parquet orc avro hive hbase kafka zookeeper güvenlik kerberos izin kota anlık görüntü").split()
weights = [1 / (i + 1) ** 1.1 for i in range(len(vocab))]          # Zipf benzeri: az kelime çok tekrar eder

with open(f"{LAB}/corpus.txt", "w") as f:
    for _ in range(600_000):
        f.write(" ".join(random.choices(vocab, weights, k=random.randint(8, 16))) + ".\n")

urls = ["/", "/urun/1", "/urun/2", "/urun/3", "/sepet", "/odeme", "/api/arama", "/api/oneri", "/giris", "/static/app.js"]
url_w = [30, 12, 10, 8, 9, 4, 14, 6, 5, 20]
status = [200, 200, 200, 200, 200, 200, 304, 301, 404, 500, 503]
t0 = datetime.datetime(2026, 10, 10)
with open(f"{LAB}/access.log", "w") as f:
    for i in range(400_000):
        u = random.choices(urls, url_w)[0]
        st = random.choice(status)
        size = 0 if st in (304, 301) else int(random.lognormvariate(8 if "static" in u else 6.5, 0.8))
        ts = (t0 + datetime.timedelta(seconds=i * 0.2)).strftime("%d/%b/%Y:%H:%M:%S")
        ip = f"10.{random.randint(0,3)}.{random.randint(0,255)}.{random.randint(1,254)}"
        f.write(f'{ip} - - [{ts} +0300] "GET {u} HTTP/1.1" {st} {size}\n')

cities = ["İstanbul", "Ankara", "İzmir", "Bursa", "Antalya"]
with open(f"{LAB}/users.csv", "w") as f:
    for uid in range(1, 2001):
        f.write(f"{uid},{random.choice(cities)}\n")
with open(f"{LAB}/orders.csv", "w") as f:
    for oid in range(1, 20001):
        f.write(f"{oid},{random.randint(1, 2100)},{round(random.uniform(10, 900), 2)}\n")   # bazı kullanıcılar yok → inner join dışında kalır

os.makedirs(f"{LAB}/small", exist_ok=True)
for i in range(1500):
    with open(f"{LAB}/small/sensor_{i:04d}.json", "w") as f:
        f.write(f'{{"sensor": {i}, "temp": {round(random.uniform(15, 35), 1)}}}\n')

for name in ["corpus.txt", "access.log", "users.csv", "orders.csv"]:
    print(f"{name:12s} {os.path.getsize(f'{LAB}/{name}') / 1024**2:7.1f} MB")
print("örnek log satırı:", open(f"{LAB}/access.log").readline().strip())

## 2. HDFS Temelleri
### 2.1 Dosya sistemi komutları
`hdfs dfs` komutları POSIX'e benzer. Her komut ayrı bir JVM başlattığı için birkaç saniye sürebilir.

In [ ]:
%%bash
hdfs dfs -mkdir -p /lab/raw /lab/logs /lab/join /lab/out
hdfs dfs -put -f $LAB/corpus.txt  /lab/raw/
hdfs dfs -put -f $LAB/access.log  /lab/logs/
hdfs dfs -put -f $LAB/users.csv $LAB/orders.csv /lab/join/

echo "--- ls -R ---";            hdfs dfs -ls -R /lab
echo "--- du -h ---";            hdfs dfs -du -h /lab
echo "--- head (ilk 2 satır) ---"; hdfs dfs -cat /lab/join/users.csv | head -2
echo "--- count (dizin, dosya, bayt) ---"; hdfs dfs -count -h /lab
echo "--- stat ---";             hdfs dfs -stat "boyut=%b blok=%o replika=%r sahip=%u değişim=%y" /lab/raw/corpus.txt

### 2.2 Bloklar: dosya nasıl bölünür?
Aynı dosyayı **16 MB** blok boyutuyla bir kez daha yazıp `fsck` ile blok listesini inceleyelim. ~45 MB'lık dosya varsayılan 128 MB blokta **1 blok**, 16 MB blokta **3 blok** olur.

In [ ]:
%%bash
hdfs dfs -D dfs.blocksize=16777216 -put -f $LAB/corpus.txt /lab/raw/corpus_16mb.txt

echo "=== 128 MB blok ==="
hdfs fsck /lab/raw/corpus.txt -files -blocks 2>/dev/null | grep -E "bytes, replicated|blk_" | head -5
echo
echo "=== 16 MB blok ==="
hdfs fsck /lab/raw/corpus_16mb.txt -files -blocks -locations 2>/dev/null | grep -E "bytes, replicated|blk_" | cut -c1-160

Blokların DataNode'un **yerel diskinde** sıradan dosyalar olarak durduğunu görelim (her bloğun yanında bir `.meta` checksum dosyası vardır):

In [ ]:
%%bash
find $HADOOP_DATA/data -name 'blk_*' -size +1M -printf "%10s bayt  %p\n" | sed "s#$HADOOP_DATA/data/##" | sort -k3 | head -8
echo "..."
echo "Toplam blok dosyası: $(find $HADOOP_DATA/data -name 'blk_*' ! -name '*.meta' | wc -l)"

## 3. HDFS İleri Konular
### 3.1 Replikasyon ve 'under-replicated' bloklar
Tek DataNode'lu kümede replikasyonu 3'e çıkarırsak NameNode bu bloğu **eksik replikalı** olarak işaretler — gerçek kümede bu durumda NameNode yeni kopyaları otomatik oluşturur.

In [ ]:
%%bash
hdfs dfs -setrep 3 /lab/join/users.csv
hdfs fsck /lab/join/users.csv 2>/dev/null | grep -iE "under-replicated|Under replicated|Status|Default replication"
hdfs dfs -setrep 1 /lab/join/users.csv > /dev/null && echo "→ replikasyon tekrar 1'e alındı"

### 3.2 Çöp kutusu (Trash) ve Snapshot

In [ ]:
%%bash
echo "=== Çöp kutusu: fs.trash.interval=60 dk ==="
hdfs dfs -rm /lab/raw/corpus_16mb.txt
hdfs dfs -ls -R /user/root/.Trash/Current/lab 2>/dev/null | tail -1
hdfs dfs -mv /user/root/.Trash/Current/lab/raw/corpus_16mb.txt /lab/raw/ && echo "✔ çöp kutusundan geri alındı"

echo; echo "=== Snapshot ==="
hdfs dfsadmin -allowSnapshot /lab/join
hdfs dfs -createSnapshot /lab/join gece_yedegi
hdfs dfs -rm -skipTrash /lab/join/users.csv            # 'yanlışlıkla' kalıcı silme
echo "Silindikten sonra:"; hdfs dfs -ls /lab/join
echo "Snapshot içeriği:"; hdfs dfs -ls /lab/join/.snapshot/gece_yedegi
hdfs dfs -cp /lab/join/.snapshot/gece_yedegi/users.csv /lab/join/ && echo "✔ snapshot'tan geri yüklendi"
hdfs lsSnapshottableDir

### 3.3 Kotalar (name quota ve space quota)

In [ ]:
%%bash
hdfs dfs -mkdir -p /lab/kota
hdfs dfsadmin -setQuota 3 /lab/kota             # dizinin kendisi dahil en fazla 3 ad nesnesi
for i in 1 2 3; do
  echo "dosya $i" | hdfs dfs -put - /lab/kota/f$i.txt 2>&1 | grep -o "NSQuotaExceededException.*" | cut -c1-120 || true
done
hdfs dfs -count -q -h /lab/kota
echo "(QUOTA, REM_QUOTA, SPACE_QUOTA, REM_SPACE_QUOTA, DIR, FILE, SIZE, PATH)"
hdfs dfsadmin -clrQuota /lab/kota

### 3.4 Küçük dosya problemi
Her dosya ve blok NameNode belleğinde ~150 bayt yer kaplar. NameNode'un **JMX** metriklerinden nesne sayılarını okuyup 1.500 küçük dosya yükleyince ne olduğuna bakalım. Ardından bunları tek bir **HAR (Hadoop Archive)** dosyasında toplayacağız (bu bir MapReduce işi çalıştırır).

In [ ]:
import requests, json

def nn_metrics():
    url = "http://localhost:9870/jmx?qry=Hadoop:service=NameNode,name=FSNamesystem"
    b = requests.get(url, timeout=10).json()["beans"][0]
    return {k: b[k] for k in ["FilesTotal", "BlocksTotal", "CapacityUsed"]}

before = nn_metrics()
subprocess.run("hdfs dfs -put -f $LAB/small /lab/small", shell=True, check=True)
after = nn_metrics()
print("Önce :", before)
print("Sonra:", after)
print(f"→ {after['FilesTotal'] - before['FilesTotal']} yeni ad nesnesi, {after['BlocksTotal'] - before['BlocksTotal']} yeni blok "
      f"— ama toplam veri sadece ~{(after['CapacityUsed'] - before['CapacityUsed']) / 1024:.0f} KB!")

In [ ]:
%%bash
hdfs dfs -mkdir -p /lab/archives
hadoop archive -archiveName sensors.har -p /lab small /lab/archives 2>&1 | grep -E "completed successfully|ERROR" || true
echo "--- HAR içinden okuma (har:// şeması) ---"
hdfs dfs -ls har:///lab/archives/sensors.har/small | head -4
hdfs dfs -cat har:///lab/archives/sensors.har/small/sensor_0007.json
echo "--- HAR'ın HDFS'teki fiziksel dosyaları ---"
hdfs dfs -ls /lab/archives/sensors.har

### 3.5 WebHDFS REST API (Python)
HDFS'e HTTP üzerinden erişim: harici sistemler, Python betikleri, dil bağımsız entegrasyonlar için.

In [ ]:
WEB = "http://localhost:9870/webhdfs/v1"
USER = {"user.name": "root"}

st = requests.get(f"{WEB}/lab/raw?op=LISTSTATUS", params=USER).json()["FileStatuses"]["FileStatus"]
for s in st:
    print(f"{s['pathSuffix']:18s} {s['length']/1024**2:8.1f} MB  blok={s['blockSize']//1024**2} MB  replika={s['replication']}")

summary = requests.get(f"{WEB}/lab?op=GETCONTENTSUMMARY", params=USER).json()["ContentSummary"]
print("\n/lab özeti:", {k: summary[k] for k in ["directoryCount", "fileCount", "length"]})

# Dosya yazma: 2 adımlı (NameNode → DataNode yönlendirmesi)
try:
    r = requests.put(f"{WEB}/lab/webhdfs_test.txt?op=CREATE&overwrite=true", params=USER, allow_redirects=False)
    loc = r.headers["Location"]
    requests.put(loc, data="WebHDFS ile yazıldı 🚀\n".encode())
    print("\nOkuma:", requests.get(f"{WEB}/lab/webhdfs_test.txt?op=OPEN", params=USER).content.decode())
except Exception as e:
    print("DataNode yönlendirmesi bu ortamda çözülemedi:", e)

## 4. MapReduce
### 4.1 Hazır örnek işler
Hadoop ile gelen `hadoop-mapreduce-examples` jar'ında `pi`, `wordcount`, `terasort` gibi örnekler vardır. İş çalışırken **8088** arayüzünde uygulamanın `ACCEPTED → RUNNING → FINISHED` geçişini izleyin.

In [ ]:
%%bash
EX=$(ls $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar)
echo "Monte Carlo ile π tahmini (4 map görevi):"
yarn jar $EX pi 4 2000 2>/dev/null | tail -2

In [ ]:
%%bash
EX=$(ls $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar)
hdfs dfs -rm -r -f -skipTrash /lab/out/wc_example > /dev/null
yarn jar $EX wordcount /lab/raw/corpus_16mb.txt /lab/out/wc_example 2>&1 \
  | grep -E "Launched map tasks|Launched reduce tasks|Map input records|Map output records|Combine input records|Combine output records|Reduce input records|Spilled Records|Shuffled Maps"
echo "--- En sık 10 kelime ---"
hdfs dfs -cat /lab/out/wc_example/part-r-* | sort -k2 -nr | head -10

👆 Counter'lara dikkat: 16 MB bloklu dosya **3 map görevi** başlattı (1 split ≈ 1 blok). `Combine input records` milyonlarca iken `Combine output records` yalnızca birkaç yüz — **combiner** shuffle'a giden veriyi binlerce kat azalttı.

### 4.2 Kendi Java WordCount'umuz
Combiner, özel bir **counter** ve **2 reducer** ile. Varsayılan `HashPartitioner` kelimeleri iki çıktı dosyasına (`part-r-00000`, `part-r-00001`) dağıtır.

In [ ]:
%%writefile /content/WordCount.java
import java.io.IOException;
import java.util.StringTokenizer;
import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.IntWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.Mapper;
import org.apache.hadoop.mapreduce.Reducer;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;

public class WordCount {

  public enum Sayac { UZUN_KELIME }          // özel counter: iş sonunda raporlanır

  public static class TokenizerMapper extends Mapper<Object, Text, Text, IntWritable> {
    private static final IntWritable ONE = new IntWritable(1);
    private final Text word = new Text();

    @Override
    public void map(Object key, Text value, Context ctx) throws IOException, InterruptedException {
      String line = value.toString().toLowerCase().replaceAll("[^\\p{L}\\s]", " ");
      StringTokenizer itr = new StringTokenizer(line);
      while (itr.hasMoreTokens()) {
        String w = itr.nextToken();
        if (w.length() >= 10) ctx.getCounter(Sayac.UZUN_KELIME).increment(1);
        word.set(w);
        ctx.write(word, ONE);
      }
    }
  }

  public static class IntSumReducer extends Reducer<Text, IntWritable, Text, IntWritable> {
    private final IntWritable result = new IntWritable();

    @Override
    public void reduce(Text key, Iterable<IntWritable> values, Context ctx)
        throws IOException, InterruptedException {
      int sum = 0;
      for (IntWritable v : values) sum += v.get();
      result.set(sum);
      ctx.write(key, result);
    }
  }

  public static void main(String[] args) throws Exception {
    Configuration conf = new Configuration();
    Job job = Job.getInstance(conf, "turkce wordcount");
    job.setJarByClass(WordCount.class);
    job.setMapperClass(TokenizerMapper.class);
    job.setCombinerClass(IntSumReducer.class);     // toplama birleşmeli + değişmeli → combiner güvenli
    job.setReducerClass(IntSumReducer.class);
    job.setNumReduceTasks(2);
    job.setOutputKeyClass(Text.class);
    job.setOutputValueClass(IntWritable.class);
    FileInputFormat.addInputPath(job, new Path(args[0]));
    FileOutputFormat.setOutputPath(job, new Path(args[1]));
    boolean ok = job.waitForCompletion(true);
    System.out.println("UZUN_KELIME counter = " +
        job.getCounters().findCounter(Sayac.UZUN_KELIME).getValue());
    System.exit(ok ? 0 : 1);
  }
}

In [ ]:
%%bash
cd /content
rm -rf wc_classes && mkdir wc_classes
javac -encoding UTF-8 -cp "$(hadoop classpath)" -d wc_classes WordCount.java && jar cf wc.jar -C wc_classes . && echo "✔ derlendi: wc.jar"

hdfs dfs -rm -r -f -skipTrash /lab/out/wc_java > /dev/null
yarn jar wc.jar WordCount /lab/raw/corpus.txt /lab/out/wc_java 2>&1 | grep -E "UZUN_KELIME|Launched reduce|completed successfully"
echo "--- Çıktı dosyaları (reducer başına bir tane) ---"
hdfs dfs -ls /lab/out/wc_java
for f in 00000 00001; do echo "part-r-$f: $(hdfs dfs -cat /lab/out/wc_java/part-r-$f | wc -l) farklı kelime"; done

## 5. Hadoop Streaming (Python ile MapReduce)
Streaming, mapper ve reducer olarak **stdin/stdout** ile çalışan herhangi bir programı kullanmanızı sağlar. Mantık şudur:

```
cat girdi | mapper | sort | reducer      ← MapReduce'un özü
```

### 5.1 Görev: URL başına istek sayısı ve ortalama yanıt boyutu

In [ ]:
%%writefile /content/log_mapper.py
#!/usr/bin/env python3
# Girdi : 10.1.2.3 - - [10/Oct/2026:00:00:00 +0300] "GET /sepet HTTP/1.1" 200 1534
# Çıktı : url \t bayt \t 1
import sys, re
pat = re.compile(r'"(?:GET|POST|PUT|DELETE) (\S+) [^"]*" (\d{3}) (\d+)')
for line in sys.stdin:
    m = pat.search(line)
    if m:
        url, status, size = m.groups()
        print(f"{url}\t{size}\t1")

In [ ]:
%%writefile /content/log_combiner.py
#!/usr/bin/env python3
# DOĞRU combiner: ortalamayı DEĞİL, (toplam, adet) çiftini taşır.
import sys
cur, total, n = None, 0, 0
for line in sys.stdin:
    key, s, c = line.rstrip("\n").split("\t")
    if key != cur:
        if cur is not None:
            print(f"{cur}\t{total}\t{n}")
        cur, total, n = key, 0, 0
    total += int(s); n += int(c)
if cur is not None:
    print(f"{cur}\t{total}\t{n}")

In [ ]:
%%writefile /content/log_reducer.py
#!/usr/bin/env python3
# Girdi anahtara göre SIRALI gelir: url \t toplam_bayt \t adet
import sys
cur, total, n = None, 0, 0
def emit():
    print(f"{cur}\t{n}\t{total / n:.1f}")
for line in sys.stdin:
    key, s, c = line.rstrip("\n").split("\t")
    if key != cur:
        if cur is not None:
            emit()
        cur, total, n = key, 0, 0
    total += int(s); n += int(c)
if cur is not None:
    emit()

Önce **kümesiz**, yalnızca Unix borularıyla deneyelim — Streaming işini kümeye göndermeden önce her zaman böyle test edin:

In [ ]:
%%bash
cd /content
echo -e "url\tistek\tort_bayt"
head -100000 $LAB/access.log | python3 log_mapper.py | sort -k1,1 | python3 log_combiner.py | sort -k1,1 | python3 log_reducer.py

Şimdi aynı betikleri kümede, **combiner** ile çalıştıralım:

In [ ]:
%%bash
cd /content
hdfs dfs -rm -r -f -skipTrash /lab/out/url_stats > /dev/null
mapred streaming \
  -D mapreduce.job.name="url-istatistikleri" \
  -D mapreduce.job.reduces=1 \
  -files log_mapper.py,log_combiner.py,log_reducer.py \
  -mapper   "python3 log_mapper.py" \
  -combiner "python3 log_combiner.py" \
  -reducer  "python3 log_reducer.py" \
  -input  /lab/logs/access.log \
  -output /lab/out/url_stats 2>&1 | grep -E "Combine input records|Combine output records|Reduce input records|completed successfully|ERROR"
echo -e "\nurl\tistek\tort_bayt"
hdfs dfs -cat /lab/out/url_stats/part-* | sort -k2 -nr

### 5.2 Combiner tuzağı: ortalamanın ortalaması ≠ ortalama
Combiner olarak "kısmi ortalama hesaplayan" bir betik kullanırsak sonuç **yanlış** olur, çünkü combiner'ın kaç kez ve hangi kayıt gruplarına uygulanacağı garanti değildir. Yerel bir simülasyonla gösterelim:

In [ ]:
import statistics, random

random.seed(1)
sizes = [random.choice([50, 60, 70]) for _ in range(9)] + [5000]       # aynı URL için 10 kayıt
chunks = [sizes[:3], sizes[3:9], sizes[9:]]                           # 3 farklı map görevinin çıktısı

true_avg = sum(sizes) / len(sizes)
wrong    = statistics.mean([statistics.mean(c) for c in chunks])      # combiner ortalama döndürürse
right    = sum(sum(c) for c in chunks) / sum(len(c) for c in chunks)  # combiner (toplam, adet) döndürürse

print(f"Gerçek ortalama          : {true_avg:8.1f}")
print(f"Yanlış combiner (ort.ort): {wrong:8.1f}   ❌")
print(f"Doğru combiner (top,adet): {right:8.1f}   ✅")

### 5.3 Reduce-side join (users ⋈ orders)
Mapper her kaydı join anahtarı (`user_id`) ile **etiketler**: kullanıcı kayıtları `U`, siparişler `O`. Aynı anahtarlı tüm kayıtlar aynı reducer'a gider; reducer şehir bilgisini siparişlere ekler. Kullanıcısı olmayan siparişler (inner join) elenir.

In [ ]:
%%writefile /content/join_mapper.py
#!/usr/bin/env python3
import sys
for line in sys.stdin:
    p = line.strip().split(",")
    if len(p) == 2:                       # users.csv  → user_id,city
        print(f"{p[0]}\tU\t{p[1]}")
    elif len(p) == 3:                     # orders.csv → order_id,user_id,amount
        print(f"{p[1]}\tO\t{p[0]},{p[2]}")

In [ ]:
%%writefile /content/join_reducer.py
#!/usr/bin/env python3
# Aynı user_id'ye ait kayıtlar ardışık gelir; o anahtarın kayıtlarını tamponlayıp birleştiriyoruz.
import sys
cur, city, orders = None, None, []
def flush():
    if cur is not None and city is not None:
        for oid, amt in orders:
            print(f"{oid}\t{cur}\t{city}\t{amt}")
for line in sys.stdin:
    uid, tag, val = line.rstrip("\n").split("\t")
    if uid != cur:
        flush()
        cur, city, orders = uid, None, []
    if tag == "U":
        city = val
    else:
        orders.append(val.split(","))
flush()

In [ ]:
%%bash
cd /content
hdfs dfs -rm -r -f -skipTrash /lab/out/joined > /dev/null
mapred streaming \
  -D mapreduce.job.name="reduce-side-join" \
  -D mapreduce.job.reduces=2 \
  -files join_mapper.py,join_reducer.py \
  -mapper "python3 join_mapper.py" -reducer "python3 join_reducer.py" \
  -input /lab/join/users.csv -input /lab/join/orders.csv \
  -output /lab/out/joined 2>&1 | grep -E "Map input records|Reduce output records|completed successfully|ERROR"

echo -e "\n--- Örnek birleşik kayıtlar (order_id, user_id, şehir, tutar) ---"
hdfs dfs -cat /lab/out/joined/part-* | head -5
echo -e "\n--- Şehir bazında ciro (awk ile son toplama) ---"
hdfs dfs -cat /lab/out/joined/part-* | awk -F'\t' '{s[$3]+=$4; n[$3]++} END {for (c in s) printf "%-10s %6d sipariş  %12.2f TL\n", c, n[c], s[c]}' | sort -k3 -nr

## 6. YARN

In [ ]:
%%bash
echo "=== Düğümler ==="; yarn node -list -all 2>/dev/null
echo; echo "=== Kuyruklar ==="
for q in default etl; do yarn queue -status $q 2>/dev/null | grep -E "Queue Name|Capacity|State" | tr '\n' ' '; echo; done
echo; echo "=== Biten uygulamalar ==="
yarn application -list -appStates FINISHED,FAILED,KILLED 2>/dev/null \
  | awk -F'\t' 'NR>2 {gsub(/ +/, " "); printf "%-32s %-26s %-12s %s\n", $1, $2, $5, $7}'

İşi belirli bir **kuyruğa** gönderelim ve aggregated loglarına bakalım:

In [ ]:
%%bash
EX=$(ls $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar)
yarn jar $EX pi -D mapreduce.job.queuename=etl 2 1000 2>/dev/null | tail -1
APP=$(yarn application -list -appStates FINISHED 2>/dev/null | awk -F'\t' '$5 ~ /etl/ {print $1}' | tr -d ' ' | tail -1)
echo "Uygulama: $APP"
yarn application -status $APP 2>/dev/null | grep -E "Queue|State|Final-State|Aggregate Resource Allocation"
sleep 5
echo "--- yarn logs (ilk satırlar) ---"
yarn logs -applicationId $APP 2>/dev/null | grep -E "Container:|LogType:|Job .* completed" | head -8

ResourceManager **REST API** ile küme metrikleri (izleme sistemleri, dashboard'lar bunu kullanır):

In [ ]:
m = requests.get("http://localhost:8088/ws/v1/cluster/metrics").json()["clusterMetrics"]
for k in ["appsSubmitted", "appsCompleted", "appsFailed", "totalMB", "availableMB", "totalVirtualCores", "activeNodes"]:
    print(f"{k:20s} {m[k]}")

apps = requests.get("http://localhost:8088/ws/v1/cluster/apps").json()["apps"]["app"]
print(f"\n{'Uygulama':28s} {'Tür':10s} {'Kuyruk':8s} {'Süre (sn)':>9s}  Durum")
for a in sorted(apps, key=lambda a: a["startedTime"]):
    print(f"{a['name'][:28]:28s} {a['applicationType']:10s} {a['queue']:8s} {a['elapsedTime']/1000:9.1f}  {a['finalStatus']}")

## 7. Dosya Formatları: CSV vs Parquet vs ORC
Aynı veriyi farklı formatlarda yazıp boyutlarını ve okuma maliyetini karşılaştırıyoruz.

In [ ]:
import pandas as pd, numpy as np, pyarrow as pa, pyarrow.parquet as pq, pyarrow.orc as orc

n = 1_000_000
rng = np.random.default_rng(0)
df = pd.DataFrame({
    "order_id": np.arange(n),
    "user_id": rng.integers(1, 50_000, n),
    "city": rng.choice(cities, n),
    "status": rng.choice(["COMPLETED", "CANCELLED", "REFUNDED"], n, p=[.88, .08, .04]),
    "amount": rng.uniform(10, 1000, n).round(2),
})
FMT = f"{LAB}/formats"; os.makedirs(FMT, exist_ok=True)
tbl = pa.Table.from_pandas(df, preserve_index=False)

df.to_csv(f"{FMT}/orders.csv", index=False)
pq.write_table(tbl, f"{FMT}/orders_snappy.parquet", compression="snappy")
pq.write_table(tbl, f"{FMT}/orders_zstd.parquet",   compression="zstd")
orc.write_table(tbl, f"{FMT}/orders_zstd.orc",      compression="zstd")

base = os.path.getsize(f"{FMT}/orders.csv")
for fn in sorted(os.listdir(FMT)):
    s = os.path.getsize(f"{FMT}/{fn}")
    print(f"{fn:24s} {s/1024**2:7.2f} MB   CSV'nin %{100*s/base:5.1f}'i")

# Sütunsal formatın gücü: yalnızca 1 sütun okumak
t = time.time(); pd.read_csv(f"{FMT}/orders.csv", usecols=["amount"]); t_csv = time.time() - t
t = time.time(); pq.read_table(f"{FMT}/orders_zstd.parquet", columns=["amount"]); t_pq = time.time() - t
print(f"\nTek sütun okuma → CSV: {t_csv:.2f} sn | Parquet: {t_pq:.3f} sn")

md_ = pq.ParquetFile(f"{FMT}/orders_zstd.parquet").metadata
print(f"Parquet metadata: {md_.num_row_groups} row group, amount sütunu min/max istatistiği:",
      md_.row_group(0).column(4).statistics.min, "/", md_.row_group(0).column(4).statistics.max)

subprocess.run(f"hdfs dfs -mkdir -p /lab/formats && hdfs dfs -put -f {FMT}/* /lab/formats/", shell=True, check=True)
print(subprocess.run("hdfs dfs -du -h /lab/formats", shell=True, capture_output=True, text=True).stdout)

## 8. Köprü: Spark ile HDFS
Spark, MapReduce'un yerini aldı ama HDFS'i ve YARN'ı kullanmaya devam eder. Spark'ı (yerel modda) çalıştırıp verileri `hdfs://localhost:9000` adresinden okuyup yine HDFS'e yazalım. (Bölüm 2 not defterinde Spark'ı derinlemesine işliyoruz.)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.4"], check=True)

# Spark 4 Java 17 ister: yalnızca Spark JVM'i için JAVA_HOME'u geçici olarak değiştiriyoruz
hadoop_java = os.environ["JAVA_HOME"]
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.master("local[*]").appName("hdfs-kopru")
         .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000")
         .config("spark.ui.port", "4050")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
os.environ["JAVA_HOME"] = hadoop_java      # Hadoop komutları için geri al
print("Spark", spark.version)

# 1) MapReduce WordCount'un Spark karşılığı — tek ifade
wc = (spark.read.text("hdfs://localhost:9000/lab/raw/corpus.txt")
      .select(F.explode(F.split(F.lower(F.regexp_replace("value", r"[^\p{L}\s]", " ")), r"\s+")).alias("w"))
      .filter("w != ''").groupBy("w").count().orderBy(F.desc("count")))
wc.show(5)

# 2) Streaming join sonucunu oku, Parquet olarak HDFS'e geri yaz
joined = (spark.read.option("sep", "\t").csv("/lab/out/joined")
          .toDF("order_id", "user_id", "city", "amount")
          .withColumn("amount", F.col("amount").cast("double")))
joined.groupBy("city").agg(F.count("*").alias("siparis"), F.round(F.sum("amount"), 2).alias("ciro")) \
      .orderBy(F.desc("ciro")).show()
joined.write.mode("overwrite").partitionBy("city").parquet("/lab/out/joined_parquet")
spark.stop()

In [ ]:
!hdfs dfs -ls /lab/out/joined_parquet

## 9. Kapatma

In [ ]:
%%bash
mapred --daemon stop historyserver
yarn   --daemon stop nodemanager
yarn   --daemon stop resourcemanager
hdfs   --daemon stop datanode
hdfs   --daemon stop namenode
jps | grep -v Jps || echo "Tüm Hadoop servisleri durduruldu."

## Alıştırmalar 🏋️

1. **Blok boyutu:** `corpus.txt` dosyasını 8 MB blok boyutuyla yükleyin ve Bölüm 4.1'deki `wordcount` işini bu dosyada çalıştırın. Kaç map görevi başladı? Toplam süre nasıl değişti?
2. **Combiner'sız çalıştırma:** Bölüm 5.1'deki streaming işini `-combiner` olmadan çalıştırın. `Reduce input records` ve `Reduce shuffle bytes` counter'larını karşılaştırın.
3. **HTTP hata oranı:** `access.log` için **saat başına** istek sayısı ve 5xx hata oranını hesaplayan bir streaming işi yazın (ipucu: anahtar = saat, değer = `toplam\t5xx`).
4. **Secondary sort:** Reduce-side join reducer'ı kullanıcı kaydını tamponlamak zorunda. `-D stream.num.map.output.key.fields=2` ve `KeyFieldBasedPartitioner` kullanarak `U` kaydının her zaman önce gelmesini sağlayın; böylece siparişleri tamponlamadan akıtın.
5. **Kuyruk kapasitesi:** `etl` kuyruğuna aynı anda 3 `pi` işi gönderin (`&` ile arka planda). 8088 → Scheduler sayfasında kuyruğun %40 kapasitesini nasıl aştığını (elasticity, max %80) gözlemleyin.
6. **DataNode kaybı:** `hdfs --daemon stop datanode` ile DataNode'u durdurun, `hdfs dfsadmin -report` ve `hdfs fsck /` çıktısını inceleyin, ardından tekrar başlatın. Tek replikalı bir kümede bu neden tehlikelidir?